[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_3_flight_planning/assignments/flight_plan_analysis.ipynb)

# Assignment 3A: Terrain analysis of the flight site

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

To fly safely, the planned flight path must clear every obstacle below it. Flight planning software sets the altitude relative to the takeoff location and knows nothing about the trees, buildings, and terrain along the route. Because we have both a lidar point cloud of the site and the planned path, we can compute the clearance between them in GRASS.

This notebook runs the same steps as the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_3_flight_planning/assignments/assignment_3a.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed.

## Learning objectives

- Convert a planned flight path (KML) into a raster of flight altitude
- Build a digital surface model (DSM) from a lidar point cloud with `r.in.pdal`
- Recognize when two elevation datasets use different vertical references and correct for it
- Compute and map the clearance between the flight path and the surface, and quantify where it is too small

## Outline

- Part 1: Environment setup
- Part 2: Flight path elevation raster
- Part 3: Surface model from lidar
- Part 4: Vertical reference and clearance
- Part 5: Your turn: the second flight path and the report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5).

In [ ]:
import json
import os
import subprocess
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters), which includes the `NGAT_COA` boundary and sample DSMs from earlier flights
- Two planned flight paths as KML files
- The 2013 Mid Pines lidar point cloud (LAS, about 165 MB)

All files are hosted in the course data bucket. The downloads total about 400 MB and take a minute or two on Colab.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

PROJECT_URL = "https://storage.googleapis.com/gis-course-data/gis584/grassdata/Lake_Wheeler_NCspm.zip"
KML1_URL = "https://storage.googleapis.com/gis-course-data/gis584/flight_path1.kml"
KML2_URL = "https://storage.googleapis.com/gis-course-data/gis584/flight_path2.kml"
LAS_URL = "https://storage.googleapis.com/gis-course-data/gis584/lidar-data/Midpines/mid_pines_spm_2013.las"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o flight_path1.kml {KML1_URL}
!curl -sSL -o flight_path2.kml {KML2_URL}
!curl -sSL -o mid_pines_spm_2013.las {LAS_URL}
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `flight_analysis` inside the Lake Wheeler project for our work (the `PERMANENT` mapset stays untouched), start the session there, and create a `Tools` object. From here on, `tools.v_import(...)` runs `v.import`, `tools.g_region(...)` runs `g.region`, and so on; a flag like `-e` is passed as `flags="e"`.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "flight_analysis"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session)

print(tools.g_proj(flags="p").text)
print(tools.g_list(type="raster,vector", mapset="PERMANENT", flags="m").text)

---
## Part 2: Flight path elevation raster

### Import the flight path

`v.import` reads the KML and reprojects it from WGS84 geographic coordinates into the project's NC State Plane on the fly. It prints a long note about the transformation pipeline it selected; that is informational.

Then we check the map is 3D and print its altitude range. `top` and `bottom` are the highest and lowest altitude in the file; write them down, we will come back to them in Part 4.

In [ ]:
tools.v_import(input="flight_path1.kml", output="flight_track_kml", overwrite=True)

topology = tools.v_info(map="flight_track_kml", flags="t").keyval
extent = tools.v_info(map="flight_track_kml", flags="g").keyval
print(f"3D map: {topology['map3d']}")
print(f"Altitude range in the KML: {extent['bottom']} to {extent['top']} m")

### Points along the path, then a raster

`v.to.points` places a point every 0.3 m along the line and interpolates the altitude (z) between vertices. We set the computational region to the flight path extent at 1 m resolution and rasterize the points with altitude as the cell value.

The aircraft does not fly a line: wind pushes it off track and the airframe has a wingspan. `r.grow` widens the path to 5 m (radius 2.5 m in map units, the `-m` flag), copying the altitude into the new cells.

In [ ]:
tools.v_to_points(input="flight_track_kml", output="flight_track_points", dmax=0.3, overwrite=True)

tools.g_region(vector="flight_track_kml", res=1)
tools.v_to_rast(input="flight_track_points", output="flight_path_altitude_rast", use="z", overwrite=True)

tools.r_grow(input="flight_path_altitude_rast", output="flight_path_wide", radius=2.5, flags="m", overwrite=True)

alt = tools.r_univar(map="flight_path_wide", format="json").json
print(f"Flight path raster: {alt['n']} cells, altitude {alt['min']:.2f} to {alt['max']:.2f} m")

#### Map: Flight path altitude over the NGAT COA boundary

In [ ]:
path_map = gj.Map(width=600, height=600)
path_map.d_rast(map="flight_path_wide")
path_map.d_vect(map="NGAT_COA@PERMANENT", fill_color="none", color="black", width=2)
path_map.d_legend(raster="flight_path_wide", at=(5, 40, 2, 6), title="Altitude (m)")
path_map.d_barscale(at=(60, 5))
path_map.show()

---
## Part 3: Surface model from lidar

`r.in.pdal` bins the point cloud into a raster. We keep first returns of class 1 (unclassified, mostly vegetation and buildings) and class 2 (ground) and take the **maximum** elevation in each 3 m cell, which gives the top of the canopy and roofs: a digital surface model (DSM). The `-o` flag skips the projection check (the file is already in NC State Plane meters) and `-e` extends the raster to the extent of the point cloud so the whole field is covered, not just the flight path region.

The lidar was flown in 2013. Trees have grown since; keep that in mind when judging small clearances.

In [ ]:
tools.r_in_pdal(
    input="mid_pines_spm_2013.las",
    output="mid_pines_surface_max",
    method="max",
    class_filter="1,2",
    return_filter="first",
    resolution=3,
    flags="oe",
    overwrite=True,
)

dsm = tools.r_univar(map="mid_pines_surface_max", format="json").json
print(f"DSM elevation within the flight path region: {dsm['min']:.1f} to {dsm['max']:.1f} m, mean {dsm['mean']:.1f} m")

> **If `r.in.pdal` fails** (for example, your GRASS build has no PDAL support), download the interpolated DSM instead and unpack it with `r.unpack` into a map named `mid_pines_surface_max`, then continue:
>
> ```python
> !curl -sSL -o secref_mid_pines_spm_elev.zip https://storage.googleapis.com/gis-course-data/gis584/lidar-data/Midpines/secref_mid_pines_spm_elev.zip && unzip -q -o secref_mid_pines_spm_elev.zip
> tools.r_unpack(input="mid_pines_lidar2013_dsm.pack", output="mid_pines_surface_max", flags="o", overwrite=True)
> ```

#### Map: Lidar DSM with the flight path

In [ ]:
dsm_map = gj.Map(width=600, height=600)
dsm_map.d_rast(map="mid_pines_surface_max")
dsm_map.d_vect(map="flight_track_kml", color="red", width=1)
dsm_map.d_legend(raster="mid_pines_surface_max", at=(5, 40, 2, 6), title="DSM (m)")
dsm_map.d_barscale(at=(60, 5))
dsm_map.show()

---
## Part 4: Vertical reference and clearance

### Do both datasets measure altitude from the same zero?

The lidar DSM is **orthometric**: height above the geoid (NAVD88), what most people call "elevation above sea level". Flight planning software often exports KML altitudes as height above the **WGS84 ellipsoid**, and in central North Carolina the ellipsoid sits about 33 m *below* the geoid. Subtracting one from the other without checking would put the aircraft 33 m lower than it really flies.

The takeoff point is the one place where we know where the aircraft is: on the ground. We take the start of the flight line and read both rasters there.

In [ ]:
# The start node of the line is the takeoff point.
start = tools.v_to_db(map="flight_track_kml", option="start", flags="p").text.strip().splitlines()[-1]
cat, x, y, z_first_vertex = start.split("|")
x, y = float(x), float(y)

at_takeoff = tools.r_what(
    map=["flight_path_wide", "mid_pines_surface_max"], coordinates=(x, y), format="json"
).json[0]
kml_takeoff_alt = at_takeoff["flight_path_wide"]["value"]
ground_takeoff = at_takeoff["mid_pines_surface_max"]["value"]

print(f"Takeoff point (E, N): {x:.1f}, {y:.1f}")
print(f"First vertex altitude in the KML:     {float(z_first_vertex):8.2f} m")
print(f"Flight path altitude at takeoff cell: {kml_takeoff_alt:8.2f} m")
print(f"Lidar ground (DSM) at takeoff:        {ground_takeoff:8.2f} m")
print(f"Ground minus flight path:             {ground_takeoff - kml_takeoff_alt:8.2f} m")

Read the numbers. If the flight path at takeoff is a few meters *above* the lidar ground, both use the same reference and no correction is needed. If it is **tens of meters below the ground**, the aircraft was not planned to fly underground: the KML altitude is ellipsoidal, and the gap is the geoid-ellipsoid separation at this site. We add that gap to the whole flight path to bring it into the DSM's reference.

The cell below applies the correction automatically when the gap is larger than 10 m. Keep the number it prints for your report, and check it against the roughly 33 m expected in this region.

In [ ]:
gap = ground_takeoff - kml_takeoff_alt
vertical_offset = round(gap, 2) if gap > 10 else 0.0

if vertical_offset:
    print(f"The KML altitude is below the ground at takeoff by {gap:.2f} m: treating it as ellipsoidal height.")
else:
    print(f"The KML altitude is {-gap:.2f} m above the ground at takeoff: same reference as the DSM.")
print(f"Vertical offset applied to the flight path: {vertical_offset} m")

tools.r_mapcalc(expression=f"flight_path_ortho = flight_path_wide + {vertical_offset}", overwrite=True)

### Clearance and the safety margin

Clearance is simply the corrected flight altitude minus the surface. We also flag every cell where the clearance is below a safety margin. 30 m is a reasonable starting point: it covers a decade of tree growth since the 2013 lidar, altitude hold error, and the wind pushing the aircraft down. Change `SAFETY_MARGIN` if your flight parameters justify a different value, and say why in your report.

The takeoff and landing segments will always show low clearance, because the aircraft climbs from and descends to the ground there. Look for low clearance *away* from the takeoff point.

In [ ]:
SAFETY_MARGIN = 30  # meters

tools.r_mapcalc(expression="clearance = flight_path_ortho - mid_pines_surface_max", overwrite=True)
tools.r_mapcalc(expression=f"low_clearance = if(clearance < {SAFETY_MARGIN}, 1, null())", overwrite=True)
tools.r_colors(map="clearance", color="ryg")
gs.write_command("r.colors", map="low_clearance", rules="-", stdin="1 red")

clr = tools.r_univar(map="clearance", flags="e", format="json").json
low = tools.r_univar(map="low_clearance", format="json").json
cruise = tools.r_univar(map="flight_path_ortho", flags="e", format="json").json["median"]
agl = cruise - ground_takeoff

print(f"Clearance along the path: min {clr['min']:.1f} m, median {clr['median']:.1f} m, max {clr['max']:.1f} m")
print(f"Cells below the {SAFETY_MARGIN} m margin: {low['n']} of {clr['n']} (about {low['n']} m2 at 1 m resolution)")
print(f"Median flight altitude: {cruise:.1f} m, which is {agl:.0f} m ({agl * 3.28084:.0f} ft) above the ground at takeoff")

#### Map: Clearance with low-clearance cells in red

Zoom in on the red cells: are they only at the takeoff and landing point, or also along the route? What is under them in the DSM?

In [ ]:
clr_map = gj.Map(width=600, height=600, use_region=True)  # zoom to the flight path region
clr_map.d_rast(map="mid_pines_surface_max")
clr_map.d_rast(map="clearance")
clr_map.d_rast(map="low_clearance")
clr_map.d_vect(map="NGAT_COA@PERMANENT", fill_color="none", color="black", width=2)
clr_map.d_legend(raster="clearance", at=(5, 40, 2, 6), title="Clearance (m)")
clr_map.d_barscale(at=(60, 5))
clr_map.show()

#### Interactive map

An interactive map lets you pan and zoom over the clearance raster with a basemap underneath. It needs the `folium` package.

In [ ]:
%pip install -q folium

interactive = gj.InteractiveMap(width=600)
interactive.add_raster("clearance", opacity=0.8)
interactive.add_raster("low_clearance")
interactive.add_vector("flight_track_kml")
interactive.add_layer_control()
interactive.show()

---
## Part 5: Your turn

### Exercise 1: The second flight path

**Task:** Repeat Parts 2 and 4 for `flight_path2.kml`. Use new map names (for example `flight_track2_kml`, `flight_path2_wide`, `clearance2`, `low_clearance2`) so you keep both results. The DSM from Part 3 can be reused.

**Hint:** Check the vertical reference again. The two KML files were exported from different software and do **not** necessarily use the same reference. Do not assume the offset from Part 4 applies.

In [ ]:
# Your code for Exercise 1 here

# Step 1: import and inspect
# tools.v_import(input="flight_path2.kml", output="flight_track2_kml", overwrite=True)
# extent2 = tools.v_info(map="flight_track2_kml", flags="g").keyval

# Step 2: points, region, raster, widen
# tools.v_to_points(...)
# tools.g_region(vector="flight_track2_kml", res=1)
# tools.v_to_rast(...)
# tools.r_grow(...)

# Step 3: takeoff check, offset, clearance, margin
# ...

# Step 4: map it
# ...

### Exercise 2: What does the safety margin cost?

**Task:** Rerun the clearance flagging for flight path 1 with `SAFETY_MARGIN` set to 15 m and to 50 m, and count the flagged cells each time.

**Hint:** Only the `low_clearance` map and the `r.univar` count change; you do not need to recompute `clearance`.

In [ ]:
# Your code for Exercise 2 here
# for margin in (15, 30, 50):
#     tools.r_mapcalc(expression=f"low_{margin} = if(clearance < {margin}, 1, null())", overwrite=True)
#     ...

---
## Report questions

Answer each question in the Markdown cell below it. These four answers, with the maps from this notebook, are the content of your report (see the assignment page for the submission details).

#### Question 1: Flight plan

Which of the two provided flight paths did you analyze as your own? Include a map of the path over the orthophoto or DSM, and state the flight altitude, the takeoff and landing location, and where the area sits within the COA boundary. You do not need to plan your own flight.

`Add text here with your answer to question 1`

#### Question 2: Clearance

For both flight paths, report the minimum and median clearance, the number and location of cells below your safety margin, and what is under the low-clearance cells in the DSM. Are any of them away from the takeoff and landing point?

`Add text here with your answer to question 2`

#### Question 3: Vertical reference

What vertical reference does each KML use, how did you find out, and what offset did you apply? What would the clearance map have shown if you had skipped this check? Convert the corrected flight altitude to height above ground at takeoff in feet and compare it with the Part 107 limit.

`Add text here with your answer to question 3`

#### Question 4: Discussion

The aim is to show that the flight plan can be flown safely. Describe your thought process: what did you take into account first, what problems did you encounter, and did you change your initial plan because of software limitations, legal limits, or the relief and obstacles of the area?

`Add text here with your answer to question 4`